<a href="https://colab.research.google.com/github/edwardoughton/satellite-image-analysis/blob/main/08_01_ggs416_26_03_23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/edwardoughton/satellite-image-analysis/blob/main/08_01_ggs416_26_03_23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GGS416 Week 8: From detections to GIS-ready data products

This notebook extends the Week 7 aerial object detection workflow.

We keep the same broad approach:
- get NAIP imagery from the Planetary Computer
- clip it to a study area
- tile the image
- run an aerial object detector on each tile
- combine detections back on the full scene

Then we go further:
- convert detections from tile pixels to global image pixels
- convert image pixels to geographic coordinates using the raster transform
- export detections as a GeoDataFrame, GeoJSON, and shapefile


## Install packages

Run this only if the packages are not already installed in your environment.

In [ ]:
%pip -q install pystac-client planetary-computer requests rasterio geopandas shapely pyogrio matplotlib pandas pillow ultralytics

## What is new in Week 8?

Last week, our outputs were mainly figures and pandas tables.

This week, the goal is to create analysis products that can be used in GIS software and downstream spatial workflows.

Main ideas:
- Affine transforms - let us move from pixel space to map space.
- CRS awareness - matters because raster pixels only become meaningful when tied to a coordinate reference system.
- Vector vs raster integration - lets us turn detections into polygons that can be mapped, filtered, exported, and joined with other data.

## Import the packages for basic Python work

In [ ]:
import os
import numpy as np
import pandas as pd

np.random.seed(42)

## Import the packages for display and plotting

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display

## Import the packages for geospatial work

These are the key packages for turning detections into GIS-ready outputs.

- `rasterio` reads raster imagery and stores the affine transform
- `geopandas` stores vector features with geometry and CRS metadata
- `shapely` builds polygons from the detection coordinates

In [ ]:
from pystac_client import Client
import planetary_computer as pc
import rasterio
import geopandas as gpd
from shapely.geometry import Polygon
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds

## Import our object detection package of choice

We continue using an oriented bounding box model from ultralytics trained for overhead imagery.

In [ ]:
from ultralytics import YOLO

## First set of helper functions: display an RGB image clearly

In [ ]:
def stretch_rgb(rgb_bands):
    """Convert a 3-band image from bands-first to bands-last and stretch it for display."""
    rgb_hwc = np.moveaxis(rgb_bands, 0, -1).astype("float32")
    p2 = np.nanpercentile(rgb_hwc, 2)
    p98 = np.nanpercentile(rgb_hwc, 98)
    return np.clip((rgb_hwc - p2) / (p98 - p2 + 1e-6), 0, 1)


def show_image(img, title, figsize=(7, 7)):
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")
    plt.show()

## Second set of helper functions: search for NAIP images

We need two search patterns:
1. get the single best recent image for a location
2. get multiple distinct acquisition dates for a location

In [ ]:
def bbox_overlap_area(a, b):
    """Calculate the overlap area of two lon/lat bounding boxes."""
    minx = max(a[0], b[0])
    miny = max(a[1], b[1])
    maxx = min(a[2], b[2])
    maxy = min(a[3], b[3])
    if maxx <= minx or maxy <= miny:
        return 0.0
    return (maxx - minx) * (maxy - miny)


def get_latest_naip_item(bbox, datetime="2021-01-01/2024-12-31"):
    catalog = Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    items = list(
        catalog.search(
            collections=["naip"],
            bbox=bbox,
            datetime=datetime,
            limit=12,
            method="POST",
        ).items()
    )

    if not items:
        raise ValueError("No NAIP scenes found for the requested bbox and date range.")

    latest_date = max(item.datetime for item in items if item.datetime).date()
    same_day_items = [item for item in items if item.datetime and item.datetime.date() == latest_date]
    return max(same_day_items, key=lambda item: bbox_overlap_area(bbox, item.bbox))


## Third set of helper functions: clip the image to our study area

The raster is stored in its own projected CRS, but our area of interest is in longitude and latitude.

So we:
1. transform the bounding box into the raster CRS
2. build a raster window from those projected bounds
3. read only the RGB bands we need
4. keep the clipped affine transform so pixels can later be geospatialized

In [ ]:
def read_naip_clip(item, bbox):
    href = item.assets["image"].href

    with rasterio.open(href) as src:
        minx, miny, maxx, maxy = transform_bounds("EPSG:4326", src.crs, *bbox)
        window = from_bounds(minx, miny, maxx, maxy, src.transform)
        rgb = src.read([1, 2, 3], window=window)
        clipped_transform = src.window_transform(window)
        clipped_crs = src.crs

    if rgb.shape[1] == 0 or rgb.shape[2] == 0:
        raise ValueError("The requested bbox produced an empty image. Try a different location.")

    return rgb, clipped_transform, clipped_crs

## Choose a place to analyze

We use the Houston Ship Channel because it usually contains ships, storage tanks, harbors, bridges, and vehicles.

In [ ]:
site_name = "Houston Ship Channel"
bbox = (-95.092, 29.735, -95.050, 29.770)
interesting_labels = {"ship"}

## Find the image and inspect its metadata

In [ ]:
item = get_latest_naip_item(bbox)
print("Selected site:", site_name)
print("Selected NAIP acquisition date:", item.datetime.date())
print(item.id)

## Read and display the clipped image

Now we read the image into memory, stretch it for display, and show it.

Notice the data shape before and after stretching:
- raw raster: `(bands, rows, cols)`
- display image: `(rows, cols, bands)`

In [ ]:
rgb, clipped_transform, clipped_crs = read_naip_clip(item, bbox)
rgb_display = stretch_rgb(rgb)

print("Raw raster shape:", rgb.shape)
print("Display image shape:", rgb_display.shape)
print("Clipped CRS:", clipped_crs)
print("Clipped affine transform:", clipped_transform)

show_image(rgb_display, f"NAIP clip: {site_name}")

## Next, we tile the image

Large aerial images are usually too large to process directly with a detector.

Tiling helps because:
- the detector expects image sizes near a standard range
- small targets are easier to detect if we do not shrink the entire scene too much
- overlap reduces the chance of missing objects on tile edges

That overlap creates new problems, e.g., duplicate detections. We will deal with that later in the class syllabus.

## Fourth set of helper functions: create image tiles

In [ ]:
def compute_positions(length, tile_size, overlap):
    step = max(tile_size - overlap, 1)
    if length <= tile_size:
        return [0]

    positions = list(range(0, length - tile_size + 1, step))
    if positions[-1] != length - tile_size:
        positions.append(length - tile_size)
    return positions


def make_tiles(image_hwc, tile_size=640, overlap=96):
    height, width, _ = image_hwc.shape
    y_positions = compute_positions(height, tile_size, overlap)
    x_positions = compute_positions(width, tile_size, overlap)

    tiles = []
    for y0 in y_positions:
        for x0 in x_positions:
            y1 = min(y0 + tile_size, height)
            x1 = min(x0 + tile_size, width)
            tile = image_hwc[y0:y1, x0:x1]
            tiles.append({"x0": x0, "y0": y0, "x1": x1, "y1": y1, "image": tile})
    return tiles


## Display a few example tiles

Before running the model, it is useful to inspect a few tiles and make sure they look reasonable.

In [ ]:
def plot_tiles(image_hwc, tiles, max_tiles=4):
    n = min(len(tiles), max_tiles)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]

    for ax, tile in zip(axes, tiles[:n]):
        ax.imshow(tile["image"])
        ax.set_title(f"x={tile['x0']} y={tile['y0']}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
tiles = make_tiles(rgb_display, tile_size=640, overlap=96)
print("Number of tiles:", len(tiles))
plot_tiles(rgb_display, tiles, max_tiles=4)

## Fifth set of helper functions: run YOLO OBB on each tile

The key idea is that detections are generated inside each tile, but we immediately shift them back into the full clipped image coordinate system.

That gives us:
- tile-local detections
- full-image pixel coordinates
- polygon coordinates for oriented boxes
- axis-aligned bounds that are useful for removing duplicates

This function below loops through the tiles, runs the pretrained OBB model, stores the results in a list of dictionaries, and finally converts these to a pandas table.

In [ ]:
def run_yolo_obb_on_tiles(model, tiles, conf=0.1, imgsz=1024):
    rows = []

    for tile_id, tile in enumerate(tiles):
        tile_img = (tile["image"] * 255).clip(0, 255).astype("uint8")
        result = model.predict(tile_img, conf=conf, imgsz=imgsz, verbose=False)[0]
        obb = result.obb

        if obb is None or len(obb) == 0:
            continue

        xywhr = obb.xywhr.cpu().numpy()
        classes = obb.cls.cpu().numpy().astype(int)
        scores = obb.conf.cpu().numpy()
        polygons = obb.xyxyxyxy.cpu().numpy()

        for i in range(len(obb)):
            class_id = int(classes[i])
            score = float(scores[i])
            label = model.names[class_id]
            poly = polygons[i]
            shifted_poly = [[float(x + tile["x0"]), float(y + tile["y0"])] for x, y in poly]

            xs = [pt[0] for pt in shifted_poly]
            ys = [pt[1] for pt in shifted_poly]

            rows.append({
                "tile_id": tile_id,
                "label": label,
                "confidence": score,
                "tile_x0": tile["x0"],
                "tile_y0": tile["y0"],
                "cx": float(xywhr[i][0] + tile["x0"]),
                "cy": float(xywhr[i][1] + tile["y0"]),
                "width": float(xywhr[i][2]),
                "height": float(xywhr[i][3]),
                "angle_radians": float(xywhr[i][4]),
                "polygon_pixels": shifted_poly,
                "xmin": float(min(xs)),
                "ymin": float(min(ys)),
                "xmax": float(max(xs)),
                "ymax": float(max(ys)),
            })

    return pd.DataFrame(rows)

Below is our plotting function for detected objects of interest.  

In [ ]:
def plot_obb_detections(image_hwc, detections, keep_labels=None, figsize=(10, 10), title="YOLO OBB detections"):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(image_hwc)

    if keep_labels is not None:
        detections = detections[detections["label"].isin(keep_labels)].copy()

    colors = {
        "ship": "deepskyblue",
        "harbor": "orange",
        "bridge": "cyan",
        "large vehicle": "yellow",
        "small vehicle": "magenta",
        "storage tank": "white",
    }

    for _, row in detections.iterrows():
        color = colors.get(row["label"], "red")
        polygon = np.asarray(row["polygon_pixels"])

        patch = patches.Polygon(
            polygon,
            closed=True,
            linewidth=1.4,
            edgecolor=color,
            facecolor="none",
        )
        ax.add_patch(patch)

    ax.set_title(title)
    ax.axis("off")
    plt.show()

## Load the pretrained YOLO OBB model

If the weights are not present, Ultralytics will download them automatically the first time this cell is run.

In [ ]:
model = YOLO("yolo11n-obb.pt")
print("Model task:", model.task)
print("Interesting labels:", interesting_labels)
print("Available classes:", model.names)

## Run detection and inspect the raw table

We intentionally use a fairly low confidence threshold first so that we have enough detections to inspect, clean, analyze, and review.

Remember, the geospatial information added to the pandas table is in the raster pixel space for the image.  

In [ ]:
detections = run_yolo_obb_on_tiles(model, tiles, conf=0.10, imgsz=1024)

if detections.empty:
    print("No detections were returned. Try a different site, larger tiles, or a lower confidence threshold.")
else:
    detections = detections[detections["label"].isin(interesting_labels)].copy()
    print("Relevant detections:", len(detections))
    display(detections.sort_values(["label", "confidence"], ascending=[True, False]).head(20))

## Plot the raw detections on the image

In [ ]:
if not detections.empty:
    print(detections["label"].value_counts())
    plot_obb_detections(
        rgb_display,
        detections,
        keep_labels=interesting_labels,
        figsize=(12, 12),
        title="Raw tile-based detections",
    )

## Converting detections into geospatial data

Right now the detections are still stored in pixel space.

Therefore, to make them useful in GIS, we need to:

1. keep the global pixel coordinates on the clipped image
2. use the raster affine transform to convert pixel columns and rows into map coordinates
3. build vector polygons with a CRS
4. export them as GIS file formats

Hence, you can move from pretty plots to GIS-ready data products.

The affine transformation is a way for us to convert between the pixel space coordinates and geographic coordinates.

In [ ]:
def pixel_to_map(col, row, transform):
    """Convert an image pixel column/row pair to map coordinates using the affine transform."""
    x, y = transform * (col, row)
    return float(x), float(y)


def polygon_pixels_to_map_coords(pixel_polygon, transform):
    return [pixel_to_map(col, row, transform) for col, row in pixel_polygon]


def detections_to_geodataframe(detections, transform, crs):
    rows = []

    for _, row in detections.iterrows():
        map_polygon = polygon_pixels_to_map_coords(row["polygon_pixels"], transform)
        geometry = Polygon(map_polygon)
        centroid = geometry.centroid

        rows.append({
            "label": row["label"],
            "confidence": row["confidence"],
            "tile_id": row["tile_id"],
            "cx_pixel": row["cx"],
            "cy_pixel": row["cy"],
            "xmin_pixel": row["xmin"],
            "ymin_pixel": row["ymin"],
            "xmax_pixel": row["xmax"],
            "ymax_pixel": row["ymax"],
            "centroid_x": centroid.x,
            "centroid_y": centroid.y,
            "geometry": geometry,
        })

    gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=crs)
    gdf_wgs84 = gdf.to_crs("EPSG:4326")
    gdf_wgs84["centroid_lon"] = gdf_wgs84.geometry.centroid.x
    gdf_wgs84["centroid_lat"] = gdf_wgs84.geometry.centroid.y
    return gdf_wgs84

## Convert the detections into a GeoDataFrame and export them


Earlier we created a local variable `clipped_transform` which was taken from the metadata of the NAIP image. We need this to carry out the transformation of pixel coordinates into geographic coordinates.

We export two common formats:
- GeoJSON for web maps and lightweight GIS work
- shapefile for older GIS workflows

In [ ]:
if detections.empty:
    print("No detections to geospatialize.")
else:
    detections_gdf = detections_to_geodataframe(detections, clipped_transform, clipped_crs)
    display(detections_gdf.head())

    os.makedirs("week8_outputs", exist_ok=True)
    geojson_path = os.path.join("week8_outputs", "houston_ship_channel_detections.geojson")
    geopackage_path = os.path.join("week8_outputs", "houston_ship_channel_detections.gpkg")
    shapefile_path = os.path.join("week8_outputs", "houston_ship_channel_detections.shp")

    detections_gdf.to_file(geojson_path, driver="GeoJSON")
    detections_gdf.to_file(geopackage_path, driver="GPKG")
    detections_gdf.to_file(shapefile_path, driver="ESRI Shapefile")

    print("Saved:", geojson_path)
    print("Saved:", shapefile_path)
    print("GeoDataFrame CRS:", detections_gdf.crs)

## Exercise - Create a GIS layer from your detected classes in new locations

For three heterogenous locations:

1. change the bounding box to a different airport, port, refinery, industrial site, or bridge crossing
2. rerun the workflow through the GeoDataFrame export step for a label within your interests (which fits the scene)
3. develop a `Pandas` function to filter your data at a certain confidence threshold.  
4. export both your bounding box, and the centroid of the bounding box as point (which might make more sensing for some objects), for your detections
5. open the exported GeoJSON, GPKG or shapefile in your chosen GIS software
6. write 200 words explaining whether the detections look spatially correct, whether there are any duplicates or false positives, and then your plan for removing both duplicates and false positives from your dataset.

You should also consider:

- Do the polygons fall on the real objects?
- Which classes look most trustworthy?
- Where do you see likely false positives?
- Why is CRS metadata necessary for this workflow?